# 1. Многопоточность

**Простой поток**

In [1]:
import threading
import time
import datetime

In [2]:
def worker(task_id):
    print(f"[Поток {task_id}] Начал работу")
    time.sleep(2)  # Имитация долгой задачи (например, скачивание файла)
    print(f"[Поток {task_id}] Закончил работу")

In [3]:
start_time = datetime.datetime.now()
for i in range(4):
    worker(i)
end_time = datetime.datetime.now()
print(f"Программа выполнена за {end_time - start_time}")

[Поток 0] Начал работу
[Поток 0] Закончил работу
[Поток 1] Начал работу
[Поток 1] Закончил работу
[Поток 2] Начал работу
[Поток 2] Закончил работу
[Поток 3] Начал работу
[Поток 3] Закончил работу
Программа выполнена за 0:00:08.003502


**Добавим разделение на потоки**

In [10]:
start_time = datetime.datetime.now()
threads = []
for i in range(4):
    # Создаем объект потока
    t = threading.Thread(target=worker, args=(i,))
    threads.append(t)
    t.start() # Запускаем поток

# join() заставляет главную программу ждать, пока поток t не закончится
for t in threads:
    t.join()

end_time = datetime.datetime.now()
print(f"Программа выполнена за {end_time - start_time}")

[Поток 0] Начал работу[Поток 1] Начал работу

[Поток 2] Начал работу
[Поток 3] Начал работу
[Поток 1] Закончил работу[Поток 2] Закончил работу
[Поток 0] Закончил работу
[Поток 3] Закончил работу

Программа выполнена за 0:00:05.022112


**Попробуем тяжелое вычисление**

In [11]:
def heavy_calculation(task_id):
    print(f"[Процесс {task_id}] Начал тяжелую математику")
    result = 0
    for i in range(100000000):
        result += i ** 2
    print(f"[Процесс {task_id}] Завершен")
    return result

In [12]:
start_time = datetime.datetime.now()
for i in range(4):
    heavy_calculation(i)
end_time = datetime.datetime.now()
print(f"Программа выполнена за {end_time - start_time}")

[Процесс 0] Начал тяжелую математику
[Процесс 0] Завершен
[Процесс 1] Начал тяжелую математику
[Процесс 1] Завершен
[Процесс 2] Начал тяжелую математику
[Процесс 2] Завершен
[Процесс 3] Начал тяжелую математику
[Процесс 3] Завершен
Программа выполнена за 0:02:30.314507


In [42]:
start_time = datetime.datetime.now()
threads = []
for i in range(4):
    t = threading.Thread(target=heavy_calculation, args = (i, ))
    threads.append(t)
    t.start()
for t in threads:
    t.join()

end_time = datetime.datetime.now()
print(f"Программа выполнена за {end_time - start_time}")

[Процесс 0] Начал тяжелую математику[Процесс 1] Начал тяжелую математику

[Процесс 2] Начал тяжелую математику
[Процесс 3] Начал тяжелую математику
[Процесс 2] Завершен
[Процесс 1] Завершен
[Процесс 0] Завершен
[Процесс 3] Завершен
Программа выполнена за 0:01:19.256539


# 2. Многопроцессность

In [13]:
import multiprocessing

if __name__ == '__main__':
    print("--- Запуск процессов ---")
    start_time = datetime.datetime.now()
    
    processes = []
    for i in range(4):
        # Создаем процесс (аналог Thread, но Process)
        p = multiprocessing.Process(target=heavy_calculation, args=(i,))
        processes.append(p)
        p.start()

    for p in processes:
        p.join() # Ждем завершения всех процессов
        
    end_time = datetime.datetime.now()
    print(f"Программа выполнена за {end_time - start_time}")

--- Запуск процессов ---
Программа выполнена за 0:00:00.501906


# 3. Race condition

In [16]:
def increment():
    global counter
    for _ in range(1000):
        current = counter
        time.sleep(0.00001)
        counter = current + 1

In [17]:
counter = 0 
for _ in range(4):
    increment()
print(counter)

4000


In [21]:
counter = 0

threads = [threading.Thread(target=increment) for _ in range(4)]

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Ожидали: 4000, получили: {counter}")

Ожидали: 4000, получили: 1001


In [22]:
def increment():
    global counter
    for _ in range(1000):
        with lock:
            current = counter
            time.sleep(0.00001)
            counter = current + 1

In [23]:
counter = 0
lock = threading.Lock()

threads = [threading.Thread(target=increment) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Ожидали: 4000, получили: {counter}")

Ожидали: 4000, получили: 4000


# 4. Concurrent.futures.ThreadPoolExecutor

In [19]:
import concurrent.futures

In [44]:
def worker(task_id):
    print(f"[Поток {task_id}] Начал работу")
    time.sleep(2)  # Имитация долгой задачи (например, скачивание файла)
    return f"[Поток {task_id}] Закончил работу"

In [45]:
task_ids = list(range(3))

start_time = datetime.datetime.now()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(worker, task_ids)
    for result in results:
        print(result)
end_time = datetime.datetime.now()
print(f"Программа выполнена за {end_time - start_time}")

[Поток 0] Начал работу
[Поток 1] Начал работу
[Поток 2] Начал работу
[Поток 0] Закончил работу
[Поток 1] Закончил работу
[Поток 2] Закончил работу
Программа выполнена за 0:00:02.006140


# 5. Concurrent.futures.ProcessPoolExecutor

In [67]:
def heavy_calculation(task_id):
    print(f"[Процесс {task_id}] Начал тяжелую математику")
    result = 0
    for i in range(100):
        result += i ** 2
    print(f"[Процесс {task_id}] Завершен")
    return result

In [71]:
if __name__ == '__main__':
    task_ids = list(range(3))
    print("--- ProcessPoolExecutor ---")
    start_time = datetime.datetime.now()
    
    # По умолчанию использует все доступные ядра CPU
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = executor.map(heavy_calculation, task_ids)
    end_time = datetime.datetime.now()
    print(f"Программа выполнена за {end_time - start_time}")

--- ProcessPoolExecutor ---
Программа выполнена за 0:00:00.265172


# 6. Практический пример

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import string
import re
import concurrent.futures
import multiprocessing
import time

In [3]:
def get_pages(url):

    """
    Извлекает список URL-адресов страниц с главной страницы библиотеки.

    Функция отправляет GET-запрос к указанному базовому URL, парсит HTML-код 
    и находит все ссылки внутри элементов с классом 'go'.

    Args:
        base_url (str): Базовый URL сайта (например, 'https://ilibrary.ru/').

    Returns:
        list[str]: Список полных URL-адресов найденных страниц. 
                   Возвращает пустой список, если запрос не удался или ссылки не найдены.
    """
    
    page = requests.get(url+'index.html')
    if page.status_code == 200:
        print(f"...Страница {url} доступна...")
    soup = BeautifulSoup(page.text, "html.parser")
    pages = []
    for link in soup.find_all(class_ = 'go'):
        pages.append("https://ilibrary.ru" + link.find('a').get('href'))
    return pages

def parse_page(page_url):
    """
    Скачивает и очищает текст конкретной страницы книги.

    Функция загружает страницу по URL, извлекает текстовое содержимое 
    из всех блоков с тегом 'z' и объединяет их в одну строку.

    Args:
        page_url (str): Полный URL-адрес страницы для парсинга.

    Returns:
        str: Очищенный текст страницы или сообщение об ошибке, 
             если запрос не удался или произошла ошибка парсинга.
    """
    
    try:
        page = requests.get(page_url)
        page_blocks = []
        if page.status_code == 200:
            soup = BeautifulSoup(page.text, "html.parser")
        for block in soup.find_all('z'):
            page_blocks.append(block.get_text())
        page_text = ''.join(page_blocks)
        return page_text
    except Exception as e:
        return f"Ошибка при скачивании {url}: {e}"

In [4]:
url = "https://ilibrary.ru/text/69/"

In [5]:
page_urls = get_pages(url)

...Страница https://ilibrary.ru/text/69/ доступна...


In [6]:
if __name__ == '__main__':
    print("--- Начало параллельного скачивания ---")
    start_time = time.time()

    # Создаем пул из 4 потоков
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        # executor.map автоматически распределит URLs по потокам
        pages_list = list(executor.map(parse_page, page_urls))

    total_time = time.time() - start_time
    print(f"Общее время: {total_time:.2f} сек.")

--- Начало параллельного скачивания ---
Общее время: 11.17 сек.


In [7]:
len(pages_list)

48

In [8]:
full_text = "".join(pages_list)

In [9]:
def preprocess_text(text):

    """
    Выполняет базовую очистку и нормализацию текста для подачи на вход LLM.

    Функция удаляет лишние пробелы, специальные символы и приводит текст 
    к нижнему регистру. 

    Args:
        text (str): Исходный текст для обработки.

    Returns:
        str: Очищенная и нормализованная строка.
    """
     
    # 1. Нормализация пробелов (убираем двойные пробелы, табы, переносы строк)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Удаление специальных символов
    # Оставляем только буквы, цифры и базовую пунктуацию
    text = re.sub(r'[^\w\s.,!?;:-]', '', text)
    
    # 3. Приведение к нижнему регистру (опционально, зависит от модели)
    text = text.lower()
    
    return text

In [45]:
if __name__ == '__main__':
    start_time = time.time()
    
    print(f"Начинаем обработку {len(pages_list)} страниц")
    with concurrent.futures.ProcessPoolExecutor(max_workers=4) as executor:
        # map автоматически распределит страницы по доступным ядрам CPU
        cleaned_pages = executor.map(preprocess_text, pages_list)
        
    total_time = time.time() - start_time
    
    print(f"Обработка завершена за {total_time:.2f} сек.")

Начинаем обработку 48 страниц
Обработка завершена за 0.26 сек.


In [46]:
cleaned_pages

<generator object _chain_from_iterable_of_lists at 0x000002121385CEE0>